# 02 - Control the modelling space

Choice modellers often want to ask focused questions:

- What if I only search transformations?
- What if I only switch between generic and alternative-specific tastes?
- What changes when I add covariates?
- What happens if I change covariate levels?

Delphos lets you do this by defining a smaller search space using a predefined catalogue of modelling terms.



### 1. Load a task and inspect the default space


In [24]:
import delphos as dp
from delphos.grammar import build_runtime

agent = dp.load_agent()

# The agent has its global catalogue that share across modelling problems
print(agent.catalogue.summary())

Catalogue(tasks=11, attributes=7, covariates=7, transformations=3, tastes=2)


The catalogue defines the modelling space available to Delphos for a given dataset. It combines the attributes and covariates available in the data with the modelling operations supported by the agent.

While `Delphos` can consider different transformations and taste structures for each attribute, covariates can be included through interactions with them to capture observed taste heterogeneity.

For example, the Swissmetro catalogue contains:

* Attributes: ```('ASC', 'time', 'cost', 'headway', 'seat')```
* Transformations: ```('linear', 'log', 'box_cox')```
* Taste structures: ```('generic', 'specific')```
* Covariates: ```('purpose', 'first', 'ticket', 'who', 'luggage', 'age', 'male', 'income', 'ga', 'origin', 'dest')```

In [25]:
# When working with a specific dataset, it constraints its modelling actions to the underlying modelling space of the dataset
task = dp.load_dataset("Swissmetro")

print(f"Attributes:       {task.attributes}")
print(f"Transformations:  {task.transform_names}")
print(f"Taste:            {task.taste_names}")
print(f"Covariates:       {task.covariate_names}")

Attributes:       (Attribute(id=1, name='ASC', alternative={}), Attribute(id=2, name='time', alternative={1: 'train_tt_scaled', 2: 'sm_tt_scaled', 3: 'car_tt_scaled'}), Attribute(id=3, name='cost', alternative={1: 'train_cost_scaled', 2: 'sm_cost_scaled', 3: 'car_co_scaled'}), Attribute(id=4, name='headway', alternative={1: 'train_he_scaled', 2: 'sm_he_scaled'}), Attribute(id=6, name='seat', alternative={2: 'sm_seats_scaled'}))
Transformations:  ('linear', 'log', 'box_cox')
Taste:            ('generic', 'specific')
Covariates:       ('purpose', 'first', 'ticket', 'who', 'luggage', 'age', 'male', 'income', 'ga', 'origin', 'dest')


### 2. Helper to count available actions


In [27]:
def count_task_actions(task, linear_additive=True):
    runtime = build_runtime(
        task=task,
        catalogue=agent.catalogue,
        linear_additive=linear_additive,
        device="cpu",
    )
    return int(runtime.action_space.task_mask.sum().item())

print("Default task actions:", count_task_actions(task))


Default task actions: 151


### 3. Transformation-only search

Keep tastes fixed to generic and remove covariates. Delphos can still choose linear, log, or Box-Cox transformations.


In [31]:
transformation_task = dp.configure_modelling_space(
    task,
    tastes=["generic"],
    covariates=[],
)

print("Transformations:", transformation_task.transform_names)
print("Tastes:", transformation_task.taste_names)
print("Covariates:", transformation_task.covariate_names)
print("Actions:", count_task_actions(transformation_task))

transformation_models = agent.propose(
    transformation_task,
    n_models=5,
    estimated = True
)

Transformations: ('linear', 'log', 'box_cox')
Tastes: ('generic',)
Covariates: ()
Actions: 14


In [32]:

transformation_models.to_dataframe()


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices
0,4,Swissmetro,1110_2110_3310_4210_5000_6210_7000,10,topk,0,False,None,5,"[25, 73, 9, 41, 137, 121, 217, 57, 89, 9]"
1,4,Swissmetro,1110_2210_3210_4210_5000_6210_7000,10,topk,1,False,None,5,"[121, 137, 41, 121, 73, 105, 25, 137, 121, 217]"
2,4,Swissmetro,1110_2110_3210_4310_5000_6110_7000,10,topk,2,False,None,5,"[121, 25, 105, 73, 89, 121, 73, 137, 41, 9]"
3,4,Swissmetro,1110_2210_3210_4110_5000_6210_7000,10,topk,3,False,None,5,"[25, 137, 41, 217, 25, 89, 73, 9, 105, 25]"
4,4,Swissmetro,1110_2210_3310_4210_5000_6110_7000,10,topk,4,False,None,5,"[25, 137, 9, 41, 217, 73, 121, 201, 89, 25]"


### 4. Taste-only search

Freeze transformations to linear and remove covariates. Delphos then searches generic vs alternative-specific tastes.


In [ ]:
taste_task = dp.configure_modelling_space(
    task,
    transformations=["linear"],
    covariates=[],
)

print("Transformations:", taste_task.transform_names)
print("Tastes:", taste_task.taste_names)
print("Covariates:", taste_task.covariate_names)
print("Actions:", count_task_actions(taste_task))

taste_models = agent.propose(
    taste_task,
    n_models=3,

)
taste_models.to_dataframe()


### 5. Add covariates deliberately

Start with one or two covariates, then expand. This keeps estimation manageable and makes interpretation easier.


In [ ]:
covariate_task = dp.configure_modelling_space(
    task,
    transformations=["linear", "log"],
    tastes=["generic", "specific"],
    covariates=["income", "purpose"],
)

print("Covariates:", covariate_task.covariate_names)
print("Actions:", count_task_actions(covariate_task))

covariate_models = agent.propose(
    covariate_task,
    n_models=3
)
covariate_models.to_dataframe()


### 6. Search only selected attributes

Use an attribute subset when you want a compact model family. `ASC` is attribute id 1 and is commonly kept.


In [ ]:
compact_task = dp.configure_modelling_space(
    task,
    attributes=["ASC", "time", "cost"],
    transformations=["linear", "log"],
    covariates=[],
)

print("Attributes:", compact_task.attribute_names)
print("Actions:", count_task_actions(compact_task))


### 7. Change covariate levels

Covariate levels control how many interaction parameters Apollo creates. Collapsing levels is often useful for quick experiments.


In [ ]:
collapsed_task = dp.set_covariate_levels(
    task,
    income=[1, 2],
    purpose=[1, 2],
)

for cov in collapsed_task.covariates:
    if cov.name in {"income", "purpose"}:
        print(cov.name, cov.levels)


### 8. Recommended workflow

1. Start with transformations only.
2. Add taste variation.
3. Add one covariate family at a time.
4. Estimate a small number of models.
5. Expand the search only after the simple space behaves well.
